# Baselines e MLflow

Este notebook treina **dois modelos de referência** no mesmo `X` e `y`, com **validação cruzada estratificada**, e grava tudo no MLflow. Serve para saber se um modelo mais pesado (ex.: MLP) realmente melhora em relação ao básico.

1. **`*DummyClassifier*` estratificado** — prevê a classe sorteando na proporção do churn na base. É o piso “sem aprendizado”: ROC-AUC perto de **0,5** e PR-AUC na ordem da **taxa de positivos** (veja a taxa no print da próxima seção) são o esperado. Qualquer modelo sério tem de ficar bem acima disso.
2. **Regressão logística** — combina o mesmo pré-processamento com um classificador linear; em dados tabulares costuma ser um *baseline* forte e interpretável.

Os dois compartilham o **mesmo** `ColumnTransformer` (`build_preprocessor` em `baselines.py`: mediana + *z-score* nos numéricos; moda + *one-hot* nas categóricas). *Seed* e número de *folds* vêm de `config.py` para reprodutibilidade.

Custos relativos FP/FN, limiar de partida **0,35** e a escolha de PR-AUC como métrica principal estão explicados em `docs/metricas_e_limiar.md`. Esses valores vão como *params* do MLflow para documentar o alinhamento com o negócio (o CV abaixo usa as métricas padrão do sklearn sobre probabilidades / corte interno do classificador).

**Como rodar:** deixe a pasta de trabalho na **raiz** do repositório `tech_challenge_fase_01` ou em `notebooks/` — a primeira célula de código ajusta o `sys.path` e aponta o *tracking* do MLflow para `mlruns/` na raiz.


In [1]:
# Caminho da raiz do repo (se o Jupyter abriu em notebooks/, sobe um nível)
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import mlflow
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate

from churn_prediction.baselines import (
    build_preprocessor,
    pipeline_dummy,
    pipeline_logistic,
)
from churn_prediction.config import (
    MLFLOW_EXPERIMENT_BASELINES,
    N_CV_SPLITS,
    RANDOM_SEED,
    THRESHOLD_START,
)
from churn_prediction.datasets import load_churn_telco, make_X_y

# MLflow grava runs em mlruns/ na raiz (file store local)
mlflow.set_tracking_uri(f"file:{ROOT / 'mlruns'}")
mlflow.set_experiment(MLFLOW_EXPERIMENT_BASELINES)


C:\Users\Primodeckers\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:D:\\tech-challenge-fase-one\\tech_challenge_fase_01/mlruns/358933226733374888', creation_time=1777845983710, experiment_id='358933226733374888', last_update_time=1777845983710, lifecycle_stage='active', name='telco_churn_baselines', tags={}, trace_location=None, workspace='default'>

## Dados e matriz `X` / vetor `y`

`load_churn_telco` lê o Excel, converte `Total Charges` com `pd.to_numeric(..., errors="coerce")` e **remove** linhas em que o total não é um número válido (no Telco são poucas — o código a seguir mostra a contagem antes do *drop*).

`make_X_y` monta o alvo binário (`Churn Label` == `Yes` → 1) e constrói `X` retirando as colunas listadas em `DROP_FOR_MODELING` em `datasets.py`:

- **Identificação:** `CustomerID`
- **Alvo e derivados diretos do churn:** `Churn Label`, `Churn Reason`, `Churn Value`
- **Suspeitas de *leakage* ou pós-evento:** `Churn Score`, `CLTV`
- **Localização fina:** `Lat Long`

O número de colunas de entrada aparece no `Shape X` impresso abaixo (números e categorias misturados), todas passando pelo mesmo pré-processamento nos dois modelos.


In [ ]:
path = ROOT / "data" / "raw" / "Telco_customer_churn.xlsx"
raw = pd.read_excel(path)
# Contagem no bruto (antes do drop) só para relatório / MLflow
n_bad_total_charges = pd.to_numeric(raw["Total Charges"], errors="coerce").isna().sum()

df = load_churn_telco(path)
X, y = make_X_y(df)

print("Linhas após limpar Total Charges:", len(df))
print("Linhas com Total Charges inválido (antes do drop):", int(n_bad_total_charges))
print("Shape X:", X.shape, "| positivos (churn):", y.sum(), "| taxa:", round(y.mean() * 100, 2), "%")


## Validação cruzada e MLflow

Usamos `StratifiedKFold` com embaralhamento fixo pela *seed*: em cada *fold* a proporção de churn fica parecida com a da base inteira — importante porque a classe positiva é minoria (taxa impressa acima).

**Quatro métricas** no `cross_validate` (para cada uma guardamos **média** e **desvio padrão** entre os 5 *folds* no MLflow):

| Métrica | Papel aqui |
|--------|------------|
| **ROC-AUC** | Mede discriminação em todos os cortes de probabilidade; útil para comparar modelos, mas pode parecer “boa” com base desequilibrada. |
| **PR-AUC** (*`average_precision`*) | Foco na classe churn; é a métrica principal que combinamos com o doc de negócio. |
| **F1** | Equilibra precisão e *recall* na classe positiva no corte **0,5** padrão do `predict`. |
| **Balanced accuracy** | Penaliza o modelo que só acerta a classe majoritária. |

Para cada *pipeline* abrimos um **run** (`dummy_stratified`, `logistic_regression`), registramos parâmetros de reprodutibilidade, dimensão dos dados e os custos/limiar **como metadado** alinhado a `metricas_e_limiar.md`. As métricas `roc_auc_*`, `average_precision_*`, etc. usam o fluxo padrão do sklearn (F1 com corte implícito **0,5** no `predict` por *fold*). Na **mesma** célula de código também calculamos probabilidades **out-of-fold**, aplicamos `THRESHOLD_START` vindo de `config.py` (hoje **0,35**) e gravamos no MLflow a métrica **`f1_at_threshold_start`** — é o F1 na classe churn nesse limiar, para alinhar ao doc de negócio.


In [ ]:
oof_proba = {}  # probabilidades churn out-of-fold, para a seção do limiar abaixo

cv = StratifiedKFold(
    n_splits=N_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED,
)
scoring = ["roc_auc", "average_precision", "f1", "balanced_accuracy"]

pre = build_preprocessor(X)

pipelines = {
    "dummy_stratified": pipeline_dummy(pre),
    "logistic_regression": pipeline_logistic(pre),
}

# Um run por pipeline; métricas agregadas nos folds (mean/std)
for name, pipe in pipelines.items():
    with mlflow.start_run(run_name=name):
        mlflow.log_params(
            {
                "random_seed": RANDOM_SEED,
                "cv_splits": N_CV_SPLITS,
                "dataset_file": path.name,
                "n_rows": len(X),
                "n_features_raw": X.shape[1],
                "cost_fn_relative": 2,
                "cost_fp_relative": 1,
                "threshold_start": THRESHOLD_START,
                "primary_metric": "average_precision",
            }
        )
        mlflow.log_param("rows_total_charges_invalid_raw", int(n_bad_total_charges))

        scores = cross_validate(
            pipe,
            X,
            y,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
        )
        for m in scoring:
            key = f"test_{m}"
            mlflow.log_metric(f"{m}_mean", float(scores[key].mean()))
            mlflow.log_metric(f"{m}_std", float(scores[key].std()))

        y_proba_oof = cross_val_predict(
            pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1
        )[:, 1]
        oof_proba[name] = y_proba_oof
        y_thr = (y_proba_oof >= THRESHOLD_START).astype(int)
        f1_thr = float(f1_score(y, y_thr))
        mlflow.log_metric("f1_at_threshold_start", f1_thr)

        row_cv = {m: round(scores[f"test_{m}"].mean(), 4) for m in scoring}
        print(name, row_cv, "| F1 @", THRESHOLD_START, "=", round(f1_thr, 4))

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", MLFLOW_EXPERIMENT_BASELINES)


dummy_stratified {'roc_auc': np.float64(0.484), 'average_precision': np.float64(0.2601), 'f1': np.float64(0.2429), 'balanced_accuracy': np.float64(0.484)}
logistic_regression {'roc_auc': np.float64(0.8498), 'average_precision': np.float64(0.6544), 'f1': np.float64(0.6011), 'balanced_accuracy': np.float64(0.7256)}
Tracking URI: file:d:\tech-challenge-fase-one\tech_challenge_fase_01\mlruns
Experimento: telco_churn_baselines


## Limiar de negócio e matriz de confusão

Usamos as probabilidades **out-of-fold** já guardadas em `oof_proba` (cada valor veio de um modelo que não viu aquela linha no treino). Com o limiar `THRESHOLD_START` do `config.py` (mesmo valor de `docs/metricas_e_limiar.md`), marcamos **churn** quando a probabilidade da classe positiva é **maior ou igual** a esse limiar.

O **F1** nesse corte também foi enviado ao MLflow como `f1_at_threshold_start`. Aqui só visualizamos as **matrizes de confusão** para conversar FP/FN com o grupo (FN mais “caro” no doc → aceitamos mais FP ao puxar o limiar para baixo em relação a 0,5).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, len(pipelines), figsize=(4.8 * len(pipelines), 4.2))

for ax, (name, _) in zip(axes, pipelines.items()):
    y_proba_oof = oof_proba[name]
    y_pred_thr = (y_proba_oof >= THRESHOLD_START).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y,
        y_pred_thr,
        display_labels=["No churn", "Churn"],
        ax=ax,
        colorbar=False,
        values_format="d",
    )
    ax.set_title(f"{name}\nP(churn) ≥ {THRESHOLD_START} → churn")

plt.suptitle("Matrizes de confusão (probabilidades out-of-fold)", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## Ver os *runs* no MLflow

Os *runs* vão para a pasta **`mlruns/`** na raiz do projeto (está no `.gitignore`, não sobe para o Git). Com o ambiente virtual ativo, na raiz do repositório rode:

`mlflow ui`

Abra o endereço que aparecer no terminal (em geral `http://127.0.0.1:5000`), selecione o experimento **`telco_churn_baselines`** e compare os dois *runs*: parâmetros à esquerda, métricas (`*_mean` / `*_std`) no painel.

**Dica:** se o MLflow avisar que o *file store* está *deprecated*, pode ignorar neste trabalho ou migrar depois para SQLite — o importante agora é ter *runs* reprodutíveis e comparáveis.

No painel de cada *run* aparece também **`f1_at_threshold_start`** (F1 na classe churn com probabilidades out-of-fold e o limiar `THRESHOLD_START` do `config.py`).

**Opcional:** numa célula extra dá para calcular *hash* do arquivo Excel e registrar como *tag* ou *param*, para provar exatamente qual versão dos dados foi usada naquele *run*.
